# MLflow Experiment Tracking Lab
**IE7374 - Machine Learning Operations (MLOps)**  
**Author:** Sowmya Shree Jayaram  
**Reference:** [raminmohammadi/MLOps](https://github.com/raminmohammadi/MLOps)

---

## Objective
This lab demonstrates **MLflow Experiment Tracking** — a core MLOps practice for logging, comparing, and managing ML experiments. We will:

1. Set up MLflow tracking (local SQLite backend)
2. Train 3 classifiers on the **Wine Quality** dataset
3. Log parameters, metrics, and artifacts to MLflow
4. Compare runs programmatically and via the MLflow UI
5. Register the best model in the MLflow Model Registry
6. Load the registered model and run inference

## Step 0: Install Dependencies
Run this cell once to install required packages.

In [ ]:
!pip install mlflow scikit-learn xgboost pandas numpy matplotlib seaborn --quiet

## Step 1: Import Libraries

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import mlflow
import mlflow.sklearn
import mlflow.xgboost
from mlflow.tracking import MlflowClient

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    confusion_matrix, classification_report
)
from xgboost import XGBClassifier

print(f"MLflow version: {mlflow.__version__}")

## Step 2: Configure MLflow Tracking

We use a **local SQLite database** as the backend store. In production, you would point this to a remote tracking server (e.g., on GCP, AWS, or Databricks).

**Key concept:** The tracking URI tells MLflow where to store experiment metadata (parameters, metrics, tags). The artifact root is where model files and plots are saved.

In [ ]:
# MLflow configuration
TRACKING_URI = "sqlite:///mlflow.db"
EXPERIMENT_NAME = "wine-quality-classification"

mlflow.set_tracking_uri(TRACKING_URI)

# Create or get experiment
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
if experiment is None:
    experiment_id = mlflow.create_experiment(EXPERIMENT_NAME)
    print(f"Created experiment '{EXPERIMENT_NAME}' with ID: {experiment_id}")
else:
    experiment_id = experiment.experiment_id
    print(f"Using existing experiment '{EXPERIMENT_NAME}' (ID: {experiment_id})")

mlflow.set_experiment(EXPERIMENT_NAME)
print(f"Tracking URI: {TRACKING_URI}")

## Step 3: Load and Explore the Wine Quality Dataset

The [Wine Quality dataset](https://archive.ics.uci.edu/ml/datasets/wine+quality) from UCI contains physicochemical properties of red wine samples. The target is a quality score from 3 to 8.

In [ ]:
# Load data
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv"
df = pd.read_csv(url, sep=";")

print(f"Dataset shape: {df.shape}")
print(f"\nFirst 5 rows:")
df.head()

In [ ]:
# Explore target distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Quality distribution
df['quality'].value_counts().sort_index().plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Wine Quality Distribution')
axes[0].set_xlabel('Quality Score')
axes[0].set_ylabel('Count')

# Correlation heatmap
sns.heatmap(df.corr(), annot=True, fmt='.2f', cmap='coolwarm', ax=axes[1],
            xticklabels=True, yticklabels=True, cbar_kws={'shrink': 0.8})
axes[1].set_title('Feature Correlation Matrix')

plt.tight_layout()
plt.show()

print(f"\nQuality value counts:\n{df['quality'].value_counts().sort_index()}")

## Step 4: Data Preprocessing

We perform:
1. Feature-target separation
2. Stratified train-test split (80/20)
3. Standard scaling (zero mean, unit variance)

In [ ]:
# Separate features and target
feature_names = [col for col in df.columns if col != 'quality']
X = df[feature_names].values
y = df['quality'].values

# Train-test split with stratification
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Standardize features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print(f"Train set: {X_train.shape[0]} samples")
print(f"Test set:  {X_test.shape[0]} samples")
print(f"Features:  {len(feature_names)}")
print(f"Classes:   {np.unique(y_train)}")

## Step 5: Helper Function — Confusion Matrix Plotting

In [ ]:
def plot_confusion_matrix(y_true, y_pred, model_name, save_path=None):
    """Generate and optionally save a confusion matrix plot."""
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=True)
    plt.title(f'Confusion Matrix — {model_name}')
    plt.ylabel('Actual')
    plt.xlabel('Predicted')
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=150)
    plt.show()
    plt.close()
    return save_path

## Step 6: Train Model 1 — Logistic Regression with MLflow Tracking

**Key MLflow APIs demonstrated:**
- `mlflow.start_run()` — begin a tracked run
- `mlflow.log_params()` — log hyperparameters
- `mlflow.log_metrics()` — log evaluation metrics
- `mlflow.log_artifact()` — log files (plots, data)
- `mlflow.sklearn.log_model()` — log the trained model

In [ ]:
os.makedirs('temp_artifacts', exist_ok=True)

with mlflow.start_run(run_name="LogisticRegression"):
    # Define hyperparameters
    params = {
        'C': 1.0,
        'max_iter': 1000,
        'solver': 'lbfgs',
        'random_state': 42
    }
    
    # Log parameters to MLflow
    mlflow.log_params(params)
    mlflow.set_tag('model_type', 'LogisticRegression')
    mlflow.set_tag('author', 'sowmya')
    
    # Train model
    lr_model = LogisticRegression(**params)
    lr_model.fit(X_train, y_train)
    
    # Predict and evaluate
    y_pred_lr = lr_model.predict(X_test)
    
    lr_metrics = {
        'accuracy': accuracy_score(y_test, y_pred_lr),
        'f1_weighted': f1_score(y_test, y_pred_lr, average='weighted'),
        'precision_weighted': precision_score(y_test, y_pred_lr, average='weighted'),
        'recall_weighted': recall_score(y_test, y_pred_lr, average='weighted')
    }
    
    # Log metrics to MLflow
    mlflow.log_metrics(lr_metrics)
    
    # Log confusion matrix as artifact
    cm_path = 'temp_artifacts/cm_logistic_regression.png'
    plot_confusion_matrix(y_test, y_pred_lr, 'Logistic Regression', cm_path)
    mlflow.log_artifact(cm_path)
    
    # Log the model itself
    mlflow.sklearn.log_model(lr_model, 'model')
    
    lr_run_id = mlflow.active_run().info.run_id

print(f"\nLogistic Regression Results:")
for k, v in lr_metrics.items():
    print(f"  {k}: {v:.4f}")
print(f"  Run ID: {lr_run_id}")

## Step 7: Train Model 2 — Random Forest

In [ ]:
with mlflow.start_run(run_name="RandomForest"):
    params = {
        'n_estimators': 200,
        'max_depth': 15,
        'min_samples_split': 5,
        'min_samples_leaf': 2,
        'random_state': 42,
        'n_jobs': -1
    }
    
    mlflow.log_params(params)
    mlflow.set_tag('model_type', 'RandomForest')
    mlflow.set_tag('author', 'sowmya')
    
    # Train
    rf_model = RandomForestClassifier(**params)
    rf_model.fit(X_train, y_train)
    
    # Evaluate
    y_pred_rf = rf_model.predict(X_test)
    
    rf_metrics = {
        'accuracy': accuracy_score(y_test, y_pred_rf),
        'f1_weighted': f1_score(y_test, y_pred_rf, average='weighted'),
        'precision_weighted': precision_score(y_test, y_pred_rf, average='weighted'),
        'recall_weighted': recall_score(y_test, y_pred_rf, average='weighted')
    }
    
    mlflow.log_metrics(rf_metrics)
    
    # Confusion matrix
    cm_path = 'temp_artifacts/cm_random_forest.png'
    plot_confusion_matrix(y_test, y_pred_rf, 'Random Forest', cm_path)
    mlflow.log_artifact(cm_path)
    
    # Feature importance plot
    importances = rf_model.feature_importances_
    sorted_idx = np.argsort(importances)[::-1]
    
    plt.figure(figsize=(10, 6))
    plt.bar(range(len(importances)), importances[sorted_idx], align='center')
    plt.xticks(range(len(importances)), [feature_names[i] for i in sorted_idx], rotation=45, ha='right')
    plt.title('Random Forest — Feature Importance')
    plt.ylabel('Importance')
    plt.tight_layout()
    fi_path = 'temp_artifacts/feature_importance_rf.png'
    plt.savefig(fi_path, dpi=150)
    plt.show()
    plt.close()
    mlflow.log_artifact(fi_path)
    
    # Log model
    mlflow.sklearn.log_model(rf_model, 'model')
    
    rf_run_id = mlflow.active_run().info.run_id

print(f"\nRandom Forest Results:")
for k, v in rf_metrics.items():
    print(f"  {k}: {v:.4f}")
print(f"  Run ID: {rf_run_id}")

## Step 8: Train Model 3 — XGBoost

In [ ]:
with mlflow.start_run(run_name="XGBoost"):
    params = {
        'n_estimators': 200,
        'max_depth': 6,
        'learning_rate': 0.1,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'random_state': 42,
        'eval_metric': 'mlogloss'
    }
    
    mlflow.log_params(params)
    mlflow.set_tag('model_type', 'XGBoost')
    mlflow.set_tag('author', 'sowmya')
    
    # Train
    xgb_model = XGBClassifier(**params)
    xgb_model.fit(X_train, y_train)
    
    # Evaluate
    y_pred_xgb = xgb_model.predict(X_test)
    
    xgb_metrics = {
        'accuracy': accuracy_score(y_test, y_pred_xgb),
        'f1_weighted': f1_score(y_test, y_pred_xgb, average='weighted'),
        'precision_weighted': precision_score(y_test, y_pred_xgb, average='weighted'),
        'recall_weighted': recall_score(y_test, y_pred_xgb, average='weighted')
    }
    
    mlflow.log_metrics(xgb_metrics)
    
    # Confusion matrix
    cm_path = 'temp_artifacts/cm_xgboost.png'
    plot_confusion_matrix(y_test, y_pred_xgb, 'XGBoost', cm_path)
    mlflow.log_artifact(cm_path)
    
    # Feature importance
    importances = xgb_model.feature_importances_
    sorted_idx = np.argsort(importances)[::-1]
    
    plt.figure(figsize=(10, 6))
    plt.bar(range(len(importances)), importances[sorted_idx], align='center', color='darkorange')
    plt.xticks(range(len(importances)), [feature_names[i] for i in sorted_idx], rotation=45, ha='right')
    plt.title('XGBoost — Feature Importance')
    plt.ylabel('Importance')
    plt.tight_layout()
    fi_path = 'temp_artifacts/feature_importance_xgb.png'
    plt.savefig(fi_path, dpi=150)
    plt.show()
    plt.close()
    mlflow.log_artifact(fi_path)
    
    # Log model
    mlflow.xgboost.log_model(xgb_model, 'model')
    
    xgb_run_id = mlflow.active_run().info.run_id

print(f"\nXGBoost Results:")
for k, v in xgb_metrics.items():
    print(f"  {k}: {v:.4f}")
print(f"  Run ID: {xgb_run_id}")

## Step 9: Compare All Runs Programmatically

Instead of using the MLflow UI, we can query runs using the `MlflowClient` API. This is useful for CI/CD pipelines where you need to programmatically select the best model.

In [ ]:
client = MlflowClient(tracking_uri=TRACKING_URI)

# Search all runs, sorted by accuracy
experiment = client.get_experiment_by_name(EXPERIMENT_NAME)
runs = client.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=['metrics.accuracy DESC']
)

# Build comparison dataframe
comparison_data = []
for run in runs:
    comparison_data.append({
        'Model': run.info.run_name,
        'Run ID': run.info.run_id[:8] + '...',
        'Accuracy': round(run.data.metrics.get('accuracy', 0), 4),
        'F1 Score': round(run.data.metrics.get('f1_weighted', 0), 4),
        'Precision': round(run.data.metrics.get('precision_weighted', 0), 4),
        'Recall': round(run.data.metrics.get('recall_weighted', 0), 4),
    })

comparison_df = pd.DataFrame(comparison_data)
print("\n=== Experiment Run Comparison (sorted by accuracy) ===")
comparison_df

In [ ]:
# Visualize comparison
metrics_to_plot = ['Accuracy', 'F1 Score', 'Precision', 'Recall']
x = np.arange(len(comparison_df))
width = 0.2

fig, ax = plt.subplots(figsize=(12, 6))
colors = ['#2196F3', '#4CAF50', '#FF9800', '#F44336']

for i, metric in enumerate(metrics_to_plot):
    bars = ax.bar(x + i * width, comparison_df[metric], width, label=metric, color=colors[i])
    for bar, val in zip(bars, comparison_df[metric]):
        ax.text(bar.get_x() + bar.get_width() / 2., bar.get_height() + 0.005,
                f'{val:.3f}', ha='center', va='bottom', fontsize=8)

ax.set_xlabel('Model')
ax.set_ylabel('Score')
ax.set_title('Model Comparison — Wine Quality Classification')
ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(comparison_df['Model'])
ax.legend()
ax.set_ylim(0, 1.0)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## Step 10: Register Best Model in MLflow Model Registry

The **Model Registry** is a centralized model store for managing the full lifecycle of MLflow models. Models go through stages:

```
None → Staging → Production → Archived
```

This is critical for production MLOps — you always know which model is live.

In [ ]:
# Find the best run
best_run = runs[0]  # Already sorted by accuracy DESC
best_run_id = best_run.info.run_id
best_model_name = best_run.info.run_name
best_accuracy = best_run.data.metrics['accuracy']

print(f"Best model: {best_model_name}")
print(f"Run ID:     {best_run_id}")
print(f"Accuracy:   {best_accuracy:.4f}")

In [ ]:
# Register the model
MODEL_NAME = "wine-quality-best-model"
model_uri = f"runs:/{best_run_id}/model"

print(f"Registering model from: {model_uri}")
result = mlflow.register_model(model_uri=model_uri, name=MODEL_NAME)
print(f"Registered: {MODEL_NAME} v{result.version}")

In [ ]:
# Transition to Staging, then Production
print(f"Transitioning v{result.version} to Staging...")
client.transition_model_version_stage(
    name=MODEL_NAME,
    version=result.version,
    stage="Staging"
)

print(f"Promoting v{result.version} to Production...")
client.transition_model_version_stage(
    name=MODEL_NAME,
    version=result.version,
    stage="Production"
)

print(f"\n'{MODEL_NAME}' v{result.version} is now in Production!")

## Step 11: Load Registered Model and Run Inference

In a production setting, a serving service would load the model from the registry using the `models:/` URI scheme.

In [ ]:
# Load the Production model from the registry
production_model_uri = f"models:/{MODEL_NAME}/Production"
print(f"Loading model from: {production_model_uri}")

loaded_model = mlflow.pyfunc.load_model(production_model_uri)

# Sample inference on a single wine
sample_wine = np.array([[
    7.4,    # fixed acidity
    0.70,   # volatile acidity
    0.00,   # citric acid
    1.9,    # residual sugar
    0.076,  # chlorides
    11.0,   # free sulfur dioxide
    34.0,   # total sulfur dioxide
    0.9978, # density
    3.51,   # pH
    0.56,   # sulphates
    9.4     # alcohol
]])

# Scale the sample using the same scaler
sample_scaled = scaler.transform(sample_wine)
prediction = loaded_model.predict(sample_scaled)

print(f"\nSample wine features: {dict(zip(feature_names, sample_wine[0]))}")
print(f"Predicted quality: {prediction[0]}")
print("\nModel loaded from registry and serving successfully!")

In [ ]:
# Batch inference on the test set
test_predictions = loaded_model.predict(X_test)
test_accuracy = accuracy_score(y_test, test_predictions)

print(f"Batch inference on {len(X_test)} test samples")
print(f"Test accuracy: {test_accuracy:.4f}")
print(f"\nClassification Report:\n")
print(classification_report(y_test, test_predictions))

## Step 12: View MLflow UI

To launch the MLflow tracking UI and visually explore all runs:

```bash
mlflow server --backend-store-uri sqlite:///mlflow.db --default-artifact-root ./artifacts --host 127.0.0.1 --port 5000
```

Then open [http://127.0.0.1:5000](http://127.0.0.1:5000) in your browser.

In the UI you can:
- Compare runs side-by-side
- View logged parameters, metrics, and artifacts
- Download confusion matrix plots
- Browse the Model Registry and model stages

## Summary

In this lab, we demonstrated the complete **MLflow experiment tracking workflow**:

| Step | MLflow Concept | API Used |
|------|---------------|----------|
| Configure tracking | Tracking URI & Experiments | `mlflow.set_tracking_uri()`, `mlflow.create_experiment()` |
| Train models | Run management | `mlflow.start_run()` |
| Log hyperparameters | Parameter logging | `mlflow.log_params()` |
| Log evaluation scores | Metric logging | `mlflow.log_metrics()` |
| Save plots & files | Artifact logging | `mlflow.log_artifact()` |
| Save trained model | Model logging | `mlflow.sklearn.log_model()` |
| Compare runs | Programmatic search | `client.search_runs()` |
| Register model | Model Registry | `mlflow.register_model()` |
| Promote model | Stage transitions | `client.transition_model_version_stage()` |
| Serve model | Model loading | `mlflow.pyfunc.load_model()` |

These practices ensure **reproducibility**, **traceability**, and **auditability** across the ML lifecycle — key requirements for production MLOps.